source for the original data! 


In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

load OTPW data

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)

In [0]:
# List all columns to see the "Menu" of features
print(f"Total Columns: {len(df_otpw.columns)}")
# Grouping columns by their source metadata
flight_cols = [c for c in df_otpw.columns if "DEP" in c or "ARR" in c or "OP_" in c]
weather_cols = [c for c in df_otpw.columns if "Hourly" in c or "Daily" in c]
print(f"Flight-related: {len(flight_cols)} | Weather-related: {len(weather_cols)}")

The CSV loader often treats everything as a string. First, we must cast our features to numeric types to perform calculations.

In [0]:
from pyspark.sql.functions import col, when, count, avg, round

# Casting key columns to numeric types for analysis
numeric_cols = [
    'DEP_DELAY', 'DEP_DEL15', 'DISTANCE', 'origin_station_dis',
    'HourlyDryBulbTemperature', 'HourlyPrecipitation', 
    'HourlyVisibility', 'HourlyWindSpeed'
]

df_eda = df_otpw
for c in numeric_cols:
    df_eda = df_eda.withColumn(c, col(c).try_cast("float"))

# Check for nulls in the target and key weather features
df_eda.select([count(when(col(c).isNull(), c)).alias(c) for c in numeric_cols]).show()

Target Variable Analysis (Class Imbalance)
Understanding the distribution of DEP_DEL15 is critical for your future Machine Learning model.

In [0]:
# Calculate the percentage of delayed vs. on-time flights
delay_stats = df_eda.groupBy("DEP_DEL15").count() \
    .withColumn("percentage", round((col("count") / df_eda.count()) * 100, 2))

display(delay_stats)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql.functions import col, when, avg, count, regexp_replace

# 1. Clean and Prepare Data
# Remove any non-numeric characters from numeric columns before casting
numeric_cols = ['DEP_DEL15', 'HourlyPrecipitation', 'origin_station_dis']

df_cleaned = df_otpw
for c in numeric_cols:
    # Use regex to replace non-digit/dot characters with empty string
    df_cleaned = df_cleaned.withColumn(c, regexp_replace(col(c), "[^0-9.]", ""))
    df_cleaned = df_cleaned.withColumn(c, col(c).try_cast("float"))

# 2. Aggregations (Calculated in Spark, collected for plotting)
# Delay Rate by Time Block
df_time = df_cleaned.groupBy("DEP_TIME_BLK").agg(avg("DEP_DEL15").alias("delay_rate")).toPandas()
df_time = df_time.sort_values("DEP_TIME_BLK")

# Delay Rate by Presence of Precipitation
df_precip = df_cleaned.withColumn("has_precip", when(col("HourlyPrecipitation") > 0, 1).otherwise(0)) \
    .groupBy("has_precip").agg(avg("DEP_DEL15").alias("delay_rate")).toPandas()

# Distance distribution
df_dist = df_cleaned.select("origin_station_dis").toPandas()

# 3. Generating Visuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Delay vs Time
axes[0].bar(df_time['DEP_TIME_BLK'], df_time['delay_rate'], color='skyblue')
axes[0].set_title('Avg Delay Rate by Time Block')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Delay Rate')

# Plot 2: Precipitation Impact
axes[1].bar(['No Precip', 'Precip'], df_precip['delay_rate'], color=['green', 'red'])
axes[1].set_title('Delay Rate: Precip vs No Precip')
axes[1].set_ylabel('Delay Rate')

# Plot 3: Distance Histogram
axes[2].hist(df_dist['origin_station_dis'].dropna(), bins=30, color='purple', edgecolor='black')
axes[2].set_title('Distribution of Station Distances')
axes[2].set_xlabel('Distance (miles)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
#plt.savefig('eda_visuals.png')

In [0]:
from pyspark.sql import functions as F

# --- 1. DATA SIZE & SHAPE (Rubric Point 2) ---
row_count = df_otpw.count()
col_count = len(df_otpw.columns)

print("="*50)
print("1. DATA SIZE & SOURCE")
print("="*50)
print(f"Total Rows (3-Month Sample): {row_count:,}")
print(f"Total Columns: {col_count}")
print("Source: Joined OTPW Dataset (NOAA Weather + BTS Flight Data)")
print("\n")

# Define a focused subset of features for our analysis
# We mix categorical, continuous, and our target variable
focus_cols = [
    'DEP_DEL15', 'DEP_DELAY', 'DISTANCE', 
    'HourlyDryBulbTemperature', 'HourlyPrecipitation', 
    'HourlyVisibility', 'origin_station_dis'
]

# --- 2. MISSING VALUE ANALYSIS (Rubric Point 4) ---
# We must check for true NULLs, NaNs, and those pesky Empty Strings ""
print("="*50)
print("2. MISSING VALUE ANALYSIS (%)")
print("="*50)

missing_exprs = [
    F.round((F.count(F.when(F.col(c).isNull() | (F.col(c) == "") | (F.col(c).try_cast("float").isNotNull() & F.isnan(F.col(c).try_cast("float"))), c)) / row_count) * 100, 2).alias(c)
    for c in focus_cols
]

df_missing = df_otpw.select(missing_exprs)
df_missing.show()

# --- 3. QUICK & DIRTY EDA / SUMMARY STATS (Rubric Point 5) ---
print("="*50)
print("3. QUICK & DIRTY SUMMARY STATISTICS")
print("="*50)

# First, we create a temporary dataframe where we safely cast strings to floats 
# (removing metadata characters like 's' or 'T' that break the summary)
df_stats = df_otpw.select(focus_cols)
for c in focus_cols:
    df_stats = df_stats.withColumn(
        c, F.regexp_replace(F.col(c), "[^0-9.-]", "").try_cast("float")
    )

# Use Spark's built-in summary for the 5-number spread + mean/stddev
display(df_stats.summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max"))

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# --- Collect the summary stats and missing data from Cell 11 ---
stats_pdf = df_stats.summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").toPandas()
stats_pdf = stats_pdf.set_index("summary")
missing_pdf = df_missing.toPandas()

feature_cols = [c for c in stats_pdf.columns]  # all feature columns

# Convert stats to numeric
for c in feature_cols:
    stats_pdf[c] = stats_pdf[c].astype(float)

# ====================================================================
# FIGURE: 2-row layout — Missing Values + Per-Feature Distributions
# ====================================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 10),
                         gridspec_kw={'height_ratios': [1, 2.2]})

# --- Panel 1: Missing Value % Bar Chart ---
ax1 = axes[0]
missing_vals = missing_pdf.iloc[0].values.astype(float)
colors = ['#e74c3c' if v > 5 else '#f39c12' if v > 1 else '#2ecc71' for v in missing_vals]
bar_labels = [f"{v:.2f}%" for v in missing_vals]
bars = ax1.bar(feature_cols, missing_vals, color=colors, edgecolor='white', linewidth=0.8)
for bar, label in zip(bars, bar_labels):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             label, ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_ylabel('Missing %', fontsize=11)
ax1.set_title('Missing Value Analysis', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(missing_vals) * 1.4)
ax1.tick_params(axis='x', rotation=25)
ax1.axhline(y=5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
ax1.text(len(feature_cols)-0.5, 5.2, '5% threshold', fontsize=8, color='gray', ha='right')

# --- Panel 2: Box-plot style distribution (min/25/50/75/max + mean) ---
ax2 = axes[1]
x_pos = np.arange(len(feature_cols))
width = 0.55

# Normalize each feature to [0, 1] range for visual comparison
norm_stats = {}
for c in feature_cols:
    fmin = stats_pdf.loc['min', c]
    fmax = stats_pdf.loc['max', c]
    rng = fmax - fmin if fmax != fmin else 1
    norm_stats[c] = {
        'min': 0,
        '25%': (stats_pdf.loc['25%', c] - fmin) / rng,
        '50%': (stats_pdf.loc['50%', c] - fmin) / rng,
        '75%': (stats_pdf.loc['75%', c] - fmin) / rng,
        'max': 1,
        'mean': (stats_pdf.loc['mean', c] - fmin) / rng,
    }

box_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22']

for i, c in enumerate(feature_cols):
    ns = norm_stats[c]
    # IQR box
    ax2.bar(i, ns['75%'] - ns['25%'], bottom=ns['25%'], width=width,
            color=box_colors[i % len(box_colors)], alpha=0.7, edgecolor='black', linewidth=0.8)
    # Median line
    ax2.plot([i - width/2, i + width/2], [ns['50%'], ns['50%']],
            color='black', linewidth=2)
    # Whiskers
    ax2.plot([i, i], [ns['min'], ns['25%']], color='black', linewidth=1)
    ax2.plot([i, i], [ns['75%'], ns['max']], color='black', linewidth=1)
    ax2.plot([i - width/4, i + width/4], [ns['min'], ns['min']], color='black', linewidth=1)
    ax2.plot([i - width/4, i + width/4], [ns['max'], ns['max']], color='black', linewidth=1)
    # Mean diamond
    ax2.scatter(i, ns['mean'], marker='D', color='red', s=40, zorder=5)

    # Annotate actual values (min, median, max)
    ax2.text(i + width/2 + 0.08, ns['min'], f"{stats_pdf.loc['min', c]:,.1f}",
             fontsize=7, va='center', color='gray')
    ax2.text(i + width/2 + 0.08, ns['50%'], f"{stats_pdf.loc['50%', c]:,.1f}",
             fontsize=7, va='center', fontweight='bold')
    ax2.text(i + width/2 + 0.08, ns['max'], f"{stats_pdf.loc['max', c]:,.1f}",
             fontsize=7, va='center', color='gray')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(feature_cols, rotation=25, ha='right', fontsize=10)
ax2.set_ylabel('Normalized Range [0–1]', fontsize=11)
ax2.set_title('Feature Distributions (Normalized Box Plots)  ◆ = Mean', fontsize=13, fontweight='bold')
ax2.set_ylim(-0.08, 1.15)
ax2.axhline(y=0.5, color='gray', linestyle=':', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reuse df_stats (numeric-casted focus features from Cell 11)
corr_pdf = df_stats.toPandas()
corr_matrix = corr_pdf.corr()

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# Tick labels
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(corr_matrix.columns, fontsize=10)

# Annotate each cell with the correlation value
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.55 else 'black'
        ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                fontsize=10, fontweight='bold', color=color)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Pearson Correlation', fontsize=11)

ax.set_title('Feature Correlation Heatmap (Focus Variables)', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()